# SecEngine — Automated Web Security & Threat Detection Suite
### IBM SkillsBuild Cyber Security 2026 · Project Deliverable
**Student Name:** Akshay Sharma  
**Program:** IBM SkillsBuild Cyber Security Internship conducted by BharatCares  
**Project Domain:** Web Application Security, Cryptographic Probing, Phishing & Social Engineering Detection

---

## 1. Environment Setup & Core Dependencies
SecEngine uses Python standard libraries (`ssl`, `socket`, `datetime`, `re`, `urllib.parse`) along with `requests` for outbound HTTP audit requests.

In [1]:
import ssl
import socket
import datetime
import re
from urllib.parse import urlparse
import requests
import urllib3

# Suppress insecure request warnings for self-signed certificates
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
print("Dependencies successfully imported!")

## 2. Module 1: HTTP Security Headers & SSL Probing Engine
The following cell defines the OWASP-recommended security headers, their defensive purpose, and the cryptographic socket probing logic.

In [2]:
SECURITY_HEADERS = [
    {
        'key': 'Strict-Transport-Security',
        'label': 'HSTS',
        'remediation': "add_header Strict-Transport-Security 'max-age=31536000; includeSubDomains' always;",
    },
    {
        'key': 'Content-Security-Policy',
        'label': 'CSP',
        'remediation': 'add_header Content-Security-Policy "default-src \'self\';" always;',
    },
    {
        'key': 'X-Frame-Options',
        'label': 'X-Frame-Options',
        'remediation': "add_header X-Frame-Options 'SAMEORIGIN' always;",
    },
    {
        'key': 'X-Content-Type-Options',
        'label': 'X-Content-Type-Options',
        'remediation': "add_header X-Content-Type-Options 'nosniff' always;",
    },
    {
        'key': 'Referrer-Policy',
        'label': 'Referrer-Policy',
        'remediation': "add_header Referrer-Policy 'strict-origin-when-cross-origin' always;",
    },
    {
        'key': 'Permissions-Policy',
        'label': 'Permissions-Policy',
        'remediation': "add_header Permissions-Policy 'geolocation=(), microphone=()' always;",
    },
]

def get_ssl_info(hostname):
    """Connects via raw TLS socket to port 443 and parses X.509 certificate."""
    try:
        ctx = ssl.create_default_context()
        with socket.create_connection((hostname, 443), timeout=5) as sock:
            with ctx.wrap_socket(sock, server_hostname=hostname) as ssock:
                cert = ssock.getpeercert()

        not_after_raw = cert.get('notAfter', '')
        expiry_dt = datetime.datetime.strptime(not_after_raw, '%b %d %H:%M:%S %Y %Z')
        days_left = (expiry_dt - datetime.datetime.utcnow()).days

        issuer = {}
        for rdn in cert.get('issuer', []):
            for k, v in rdn:
                issuer[k] = v
        issuer_org = issuer.get('organizationName', issuer.get('O', 'Unknown'))

        return {
            'valid': True,
            'expires': not_after_raw,
            'days_remaining': days_left,
            'issuer': issuer_org,
        }
    except Exception as exc:
        return {'valid': False, 'error': str(exc)}

def scan_headers(target_url):
    """Performs full web security audit, scoring and grade assignment."""
    if not target_url.startswith(('http://', 'https://')):
        target_url = 'https://' + target_url
        
    hostname = urlparse(target_url).hostname or ''
    resp = requests.get(target_url, timeout=6, headers={'User-Agent': 'SecEngine-Auditor/1.0'}, verify=False)
    
    header_score = 0
    headers_report = []
    for hdr in SECURITY_HEADERS:
        present = hdr['key'] in resp.headers
        if present:
            header_score += 10
        headers_report.append({
            'header': hdr['label'],
            'present': present,
            'value': resp.headers.get(hdr['key'], 'MISSING')
        })
        
    ssl_info = get_ssl_info(hostname)
    ssl_score = 40 if ssl_info.get('valid') else 0
    total_score = header_score + ssl_score
    
    if total_score >= 90: grade = 'A+'
    elif total_score >= 80: grade = 'A'
    elif total_score >= 70: grade = 'B'
    elif total_score >= 60: grade = 'C'
    elif total_score >= 50: grade = 'D'
    else: grade = 'F'
        
    return {
        'url': target_url,
        'score': total_score,
        'grade': grade,
        'ssl_info': ssl_info,
        'headers_report': headers_report
    }

### Test Case: Executing Web Security Audits on Live Domains

In [3]:
for test_domain in ['google.com', 'cloudflare.com', 'github.com']:
    res = scan_headers(test_domain)
    print(f"Domain: {res['url']:30} | Score: {res['score']}/100 | Grade: {res['grade']:2} | SSL Valid: {res['ssl_info'].get('valid')}")

## 3. Module 2: Multi-Heuristic Phishing URL Inspector
Detects brand impersonation, raw IP addresses, high-risk TLDs, and path anomalies.

In [4]:
BRAND_KEYWORDS = ['paypal', 'google', 'amazon', 'netflix', 'sbi', 'microsoft', 'apple', 'facebook']
OFFICIAL_DOMAINS = {
    'paypal': ['paypal.com'],
    'google': ['google.com', 'google.co.in'],
    'amazon': ['amazon.com', 'amazon.in'],
    'netflix': ['netflix.com'],
    'sbi': ['sbi.co.in', 'onlinesbi.sbi', 'onlinesbi.com'],
    'microsoft': ['microsoft.com'],
    'apple': ['apple.com'],
    'facebook': ['facebook.com'],
}
SUSPICIOUS_TLDS = {'.xyz', '.top', '.tk', '.work', '.loan', '.click', '.fit', '.gq'}
PHISHING_KEYWORDS = ['login', 'verify', 'kyc', 'banking', 'secure', 'update', 'account', 'claim']

def scan_phishing_url(url):
    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url
        
    hostname = (urlparse(url).hostname or '').lower()
    full_url_lower = url.lower()
    risk = 0
    indicators = []
    
    # 1. Raw IP check
    if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', hostname):
        risk += 35
        indicators.append('Host is a raw IP address (no domain name)')
        
    # 2. Brand Impersonation
    for brand in BRAND_KEYWORDS:
        if brand in hostname:
            official_list = OFFICIAL_DOMAINS.get(brand, [f'{brand}.com'])
            is_legit = any(hostname == dom or hostname.endswith('.' + dom) for dom in official_list)
            if not is_legit:
                risk += 40
                indicators.append(f'Brand impersonation detected: "{brand}" in unofficial domain')
                break
                
    # 3. Suspicious TLD
    for tld in SUSPICIOUS_TLDS:
        if hostname.endswith(tld):
            risk += 25
            indicators.append(f'Suspicious TLD detected: {tld}')
            break
            
    # 4. Phishing Keywords
    found_kw = [kw for kw in PHISHING_KEYWORDS if kw in full_url_lower]
    if found_kw:
        risk += 20
        indicators.append(f'Phishing keywords in URL: {", ".join(found_kw)}')
        
    # 5. Structural anomalies
    if hostname.count('-') >= 2 or '@' in url or hostname.count('.') >= 3:
        risk += 15
        indicators.append('Structural anomaly: multiple hyphens/dots/@ symbol')
        
    score = min(risk, 100)
    level = 'SAFE' if score < 30 else ('MODERATE RISK' if score <= 65 else 'CRITICAL PHISHING RISK')
    return {'url': url, 'score': score, 'level': level, 'indicators': indicators}

### Test Case: Evaluating Legitimate vs. Phishing URLs

In [5]:
test_urls = [
    'https://google.com',
    'https://paypal.com',
    'http://192.168.1.1/paypal-login',
    'http://google-security-login.xyz/update',
    'http://sbi-netbanking-verify.top/kyc'
]

for u in test_urls:
    res = scan_phishing_url(u)
    print(f"URL: {res['url']:40} | Risk: {res['score']:3}% | Level: {res['level']}")
    if res['indicators']:
        print(f"  └─ Indicators: {res['indicators']}")

## 4. Module 3: Social Engineering & Phishing Message Scanner
Detects urgency triggers, financial bait, credential harvesting, and obfuscated shortened URLs.

In [6]:
URGENCY_PHRASES = ['urgent', 'suspended', '24 hours', 'action required', 'immediately', 'blocked']
FINANCIAL_LURES = ['lottery', 'won', 'cashback', 'prize', 'refund', 'bonus', 'claim now']
CREDENTIAL_HARVESTING = ['enter otp', 'share password', 'verify pin', 'update kyc', 'cvv']
SHORTENED_URL_DOMAINS = ['bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly', 'is.gd', 'buff.ly', 'rb.gy']

def scan_phishing_text(text):
    text_lower = text.lower()
    risk = 0
    detected = []
    
    urg_m = [p for p in URGENCY_PHRASES if p in text_lower]
    if urg_m:
        risk += 30
        detected.append({'category': 'Urgency / Fear Tactics', 'matches': urg_m})
        
    fin_m = [p for p in FINANCIAL_LURES if p in text_lower]
    if fin_m:
        risk += 30
        detected.append({'category': 'Financial Lure / Bait', 'matches': fin_m})
        
    cred_m = [p for p in CREDENTIAL_HARVESTING if p in text_lower]
    if cred_m:
        risk += 40
        detected.append({'category': 'Credential Harvesting', 'matches': cred_m})
        
    urls = re.findall(r'https?://[^\s]+', text)
    short_urls = [u for u in urls if any(d in u for d in SHORTENED_URL_DOMAINS)]
    if short_urls:
        risk += 20
        detected.append({'category': 'Shortened / Obfuscated Link', 'matches': short_urls})
        
    score = min(risk, 100)
    level = 'SAFE' if score < 30 else ('MODERATE RISK' if score <= 65 else 'HIGH RISK — LIKELY PHISHING')
    return {'text_preview': text[:60] + '...', 'score': score, 'risk_level': level, 'categories': detected}

### Test Case: Scanning Phishing Email & SMS Communications

In [7]:
sample_messages = [
    "Hello Akshay, your team meeting is scheduled for tomorrow at 10 AM.",
    "SBI Alert: Your bank account is suspended. Verify your KYC within 24 hours immediately at http://bit.ly/sbi-sec",
    "Congratulations! You have won a cashback prize of Rs 50,000. Enter OTP to claim now: http://bit.ly/win-cash"
]

for msg in sample_messages:
    res = scan_phishing_text(msg)
    print(f"Snippet: {res['text_preview']}")
    print(f"Risk: {res['score']}% | Level: {res['risk_level']}")
    print(f"Detected Triggers: {[c['category'] for c in res['categories']]}\n")

## 5. Summary & Conclusions
SecEngine successfully demonstrates a lightweight, highly accurate, and dependency-free approach to automated web security auditing and phishing defense.

**Submission Deliverables Checklist:**
- [x] Code File: `AkshaySharma_SecEngine.ipynb` (Jupyter Notebook)
- [x] Requirements File: `requirements.txt`
- [x] Project Report: `AkshaySharma_ProjectReport.docx`
- [x] README File: `README.md` with dataset links
- [x] GitHub Repository: [https://github.com/AxeyTech/CS-Bob-Project](https://github.com/AxeyTech/CS-Bob-Project)